## Alpha sweep — finding the optimal distance metric

So far we ran all distance configs at **neutral alpha** — the value that gives each feature equal weight, mimicking Gower. This is a fair starting point for comparison, but it is an arbitrary one.

Alpha controls how much the numerical block contributes vs the categorical block:
- `alpha = 1.0` → pure numerical distance, categoricals ignored
- `alpha = 0.0` → pure categorical distance, numericals ignored  
- `alpha = 0.8` → numericals dominate (our neutral baseline for Hamming)
- `alpha = 0.58` → numericals still dominate but less so (our neutral baseline for Tanimoto)

The neutral baseline just says *"I have no prior belief about which block matters more, so I weight them proportionally to how many features each has."* That is a reasonable default but it is not necessarily the best for your data.

The problem with only comparing configs at neutral alpha is that a metric which looks worse than another at neutral alpha might actually be better at a different alpha. For example L2+Hamming might beat L1+Tanimoto if you give the categorical block more weight — we simply do not know yet.

So instead of picking a winner at neutral alpha and then tuning it, we sweep alpha across all configs simultaneously. Every config gets its best possible alpha, and we compare them at their optimal. This is the only fair comparison.

We also run each combination across multiple random seeds and average the silhouette, because with only 49 observations a single kmedoids run can get lucky or unlucky on the initial medoid placement. A difference of 0.02 in silhouette on one seed is noise; a consistent difference across 5 seeds is signal.

The output will be a table of `(num_dist, cat_dist, best_alpha, best_k, mean_silhouette)` — the winner is the row with the highest mean silhouette.

In [1]:
import numpy as np
import pandas as pd
import pickle
import logging
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
import gower as gower_lib
import kmedoids
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import umap
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sys
sys.path.append(str(Path.cwd()))
from distance import (
    mixed_distance_matrix,
    check_distance_matrix,
    build_gower_cat_mask,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s",
)
logger = logging.getLogger(__name__)

c:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_metadata = pd.read_excel(
    "C:/Users/giuli/Repositories/fintech-group-work/BusinessCase1/Data/BankClients_Metadata.xlsx",
    nrows = 50
)
print(f"Metadata: {df_metadata.shape[0]:,} rows × {df_metadata.shape[1]} columns")
df_metadata.head()

Metadata: 18 rows × 2 columns


,Metadata,Unnamed: 1
0,ID,Numerical ID
1,Age,"Age, in years"
2,Gender,"Gender (Female = 1, Male = 0)"
3,Job,1 = Unemployed\n2 = Employee/Worker\n3 = Manag...
4,Area,"1 = North, 2 = Center, 3 = South/Islands"


In [3]:
df_raw = pd.read_excel(
    r"C:\Users\giuli\Repositories\fintech-group-work\BusinessCase1\Data\Dataset1_BankClients.xlsx"
)
print(f"Raw data: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head()


Raw data: 5,000 rows × 18 columns


,ID,Age,Gender,Job,Area,CitySize,FamilySize,Income,Wealth,Debt,FinEdu,ESG,Digital,BankFriend,LifeStyle,Luxury,Saving,Investments
0,1,24,1,1,2,2,4,0.668046,0.702786,0.262070,0.741853,0.483684,0.698625,0.618259,0.607877,0.897369,0.283222,1
1,2,47,1,2,2,3,1,0.858453,0.915043,0.730430,0.859423,0.537167,0.959025,0.785936,0.862271,0.913729,0.821590,3
2,3,38,0,2,1,2,2,0.926818,0.898316,0.441272,0.485953,0.649434,0.750265,0.699725,0.755404,0.765199,0.503790,3
3,4,67,0,2,1,2,3,0.538797,0.423180,0.600401,0.493144,0.533829,0.590165,0.675353,0.334432,0.517209,0.691240,2
4,5,33,0,2,1,3,1,0.806659,0.731404,0.831449,0.856286,0.784940,0.710026,0.758793,0.908878,0.611610,0.615916,2


In [4]:
# ── Column definitions ───────────────────────────────────────────────────────
categorical_cols = ["Gender", "Job", "Area"]
numerical_cols   = [c for c in df_raw.columns if c not in categorical_cols + ["ID"]]
 
logger.info("Numerical columns (%d): %s", len(numerical_cols), numerical_cols)
logger.info("Categorical columns (%d): %s", len(categorical_cols), categorical_cols)
 
# ── 1. Drop ID ───────────────────────────────────────────────────────────────
df = df_raw.drop(columns=["ID"])
 
# ── 2. Missing values ────────────────────────────────────────────────────────
for col in numerical_cols:
    n_missing = df[col].isna().sum()
    if n_missing:
        logger.info("Imputing %d missing values in '%s' with median.", n_missing, col)
    df[col] = df[col].fillna(df[col].median())
 
for col in categorical_cols:
    n_missing = df[col].isna().sum()
    if n_missing:
        logger.info("Imputing %d missing values in '%s' with mode.", n_missing, col)
    df[col] = df[col].fillna(df[col].mode()[0])
 
assert df.isna().sum().sum() == 0, "Missing values remain after imputation!"
logger.info("Missing values after imputation: 0")
 
# ── 3. Hard domain filter — working minors ───────────────────────────────────
mask_minors = (df["Age"] < 18) & (df["Job"].isin([2, 3, 4]))
n_removed   = mask_minors.sum()
df = df[~mask_minors].reset_index(drop=True)
logger.info("Minor filter: removed %d rows → %d remaining.", n_removed, len(df))
 
# ── 4. Isolation Forest — remove top 1%% multivariate outliers ───────────────
iso    = IsolationForest(contamination=0.01, random_state=42)
labels = iso.fit_predict(df[numerical_cols])
n_out  = (labels == -1).sum()
df     = df[labels == 1].reset_index(drop=True)
logger.info("Isolation Forest: removed %d outliers → %d remaining.", n_out, len(df))
 
# ── 5. Prepare input arrays ──────────────────────────────────────────────────
 
# Numerical: min-max scale to [0, 1]
scaler = MinMaxScaler()
X_num  = scaler.fit_transform(df[numerical_cols]).astype(float)
logger.info("X_num scaled to [0,1]: min=%.4f  max=%.4f", X_num.min(), X_num.max())
 
# Categorical — label-encoded (for Hamming; one integer per variable)
le_encoders = {}
X_cat_int   = np.zeros((len(df), len(categorical_cols)), dtype=np.int32)
for j, col in enumerate(categorical_cols):
    le = LabelEncoder()
    X_cat_int[:, j] = le.fit_transform(df[col])
    le_encoders[col] = le
    logger.info("LabelEncoder '%s': %d classes → %s", col, len(le.classes_), list(le.classes_))
 
# Categorical — one-hot encoded (for Tanimoto; one binary column per category level)
X_cat_ohe = pd.get_dummies(df[categorical_cols].astype(str)).values.astype(np.int32)
logger.info("One-hot encoded shape: %s  unique values: %s",
            X_cat_ohe.shape, np.unique(X_cat_ohe).tolist())
 
 
n_num = X_num.shape[1]

# Hamming: n_cat = number of original categorical variables (one column each)
alpha_neutral_hamming  = n_num / (n_num + X_cat_int.shape[1])
# Tanimoto: n_cat = number of one-hot binary columns (one per category level)
alpha_neutral_tanimoto = n_num / (n_num + X_cat_ohe.shape[1])

logger.info(
    "Neutral alpha (Hamming)  = %d / (%d + %d) = %.4f",
    n_num, n_num, X_cat_int.shape[1], alpha_neutral_hamming,
)
logger.info(
    "Neutral alpha (Tanimoto) = %d / (%d + %d) = %.4f",
    n_num, n_num, X_cat_ohe.shape[1], alpha_neutral_tanimoto,
)
 
print(f"\nX_num shape     : {X_num.shape}")
print(f"X_cat_int shape : {X_cat_int.shape}  unique={np.unique(X_cat_int).tolist()}")
print(f"X_cat_ohe shape : {X_cat_ohe.shape}  unique={np.unique(X_cat_ohe).tolist()}")
print(f"Neutral alpha (Hamming)  : {alpha_neutral_hamming:.4f}")
print(f"Neutral alpha (Tanimoto) : {alpha_neutral_tanimoto:.4f}")

2026-03-19 23:06:56,807 — INFO — Numerical columns (14): ['Age', 'CitySize', 'FamilySize', 'Income', 'Wealth', 'Debt', 'FinEdu', 'ESG', 'Digital', 'BankFriend', 'LifeStyle', 'Luxury', 'Saving', 'Investments']
2026-03-19 23:06:56,809 — INFO — Categorical columns (3): ['Gender', 'Job', 'Area']
2026-03-19 23:06:56,823 — INFO — Missing values after imputation: 0
2026-03-19 23:06:56,826 — INFO — Minor filter: removed 0 rows → 5000 remaining.
2026-03-19 23:06:57,077 — INFO — Isolation Forest: removed 50 outliers → 4950 remaining.
2026-03-19 23:06:57,082 — INFO — X_num scaled to [0,1]: min=0.0000  max=1.0000
2026-03-19 23:06:57,086 — INFO — LabelEncoder 'Gender': 2 classes → [np.int64(0), np.int64(1)]
2026-03-19 23:06:57,087 — INFO — LabelEncoder 'Job': 5 classes → [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
2026-03-19 23:06:57,088 — INFO — LabelEncoder 'Area': 3 classes → [np.int64(1), np.int64(2), np.int64(3)]
2026-03-19 23:06:57,105 — INFO — One-hot encoded shape: (49


X_num shape     : (4950, 14)
X_cat_int shape : (4950, 3)  unique=[0, 1, 2, 3, 4]
X_cat_ohe shape : (4950, 10)  unique=[0, 1]
Neutral alpha (Hamming)  : 0.8235
Neutral alpha (Tanimoto) : 0.5833


In [ ]:
# %% Cell 4 — Alpha sweep (find optimal alpha and k before computing anything)
from itertools import product

ALPHA_SWEEP = [0.2, 0.3, 0.4, 0.5, 0.6, alpha_neutral_hamming, alpha_neutral_tanimoto, 0.8, 0.9]
ALPHA_SWEEP = sorted(set(round(a, 4) for a in ALPHA_SWEEP))
K_RANGE     = range(2, 11)
SEEDS       = [42, 7, 123, 999, 2024]

sweep_configs = [
    ("L1",       "Hamming",  X_cat_int.copy()),
    ("L1",       "Tanimoto", X_cat_ohe.copy()),
    ("L2",       "Hamming",  X_cat_int.copy()),
    ("L2",       "Tanimoto", X_cat_ohe.copy()),
    ("Canberra", "Hamming",  X_cat_int.copy()),
    ("Canberra", "Tanimoto", X_cat_ohe.copy()),
]

logger.info("Alpha sweep: %s", ALPHA_SWEEP)
logger.info("Seeds: %s", SEEDS)
logger.info(
    "Configs: %d × alphas: %d × k: %d × seeds: %d = %d total runs",
    len(sweep_configs), len(ALPHA_SWEEP), len(K_RANGE), len(SEEDS),
    len(sweep_configs) * len(ALPHA_SWEEP) * len(K_RANGE) * len(SEEDS),
)

sweep_results = []

for num_dist, cat_dist, X_cat in sweep_configs:
    for alpha in ALPHA_SWEEP:
        D = mixed_distance_matrix(
            X_num, X_cat,
            alpha=alpha,
            num_dist=num_dist,
            cat_dist=cat_dist,
        ).astype(np.float64)

        for k in K_RANGE:
            sil_scores = []
            for seed in SEEDS:
                res  = kmedoids.fasterpam(D, k, random_state=seed)
                lbls = np.array(res.labels)
                if len(np.unique(lbls)) < 2:
                    continue
                sil_scores.append(silhouette_score(D, lbls, metric="precomputed"))

            if not sil_scores:
                continue

            mean_sil = float(np.mean(sil_scores))
            std_sil  = float(np.std(sil_scores))

            sweep_results.append({
                "num_dist": num_dist,
                "cat_dist": cat_dist,
                "alpha":    round(alpha, 4),
                "k":        k,
                "mean_sil": round(mean_sil, 4),
                "std_sil":  round(std_sil,  4),
            })

df_sweep = pd.DataFrame(sweep_results)

df_best = (
    df_sweep
    .sort_values("mean_sil", ascending=False)
    .groupby(["num_dist", "cat_dist"], sort=False)
    .first()
    .reset_index()
    [["num_dist", "cat_dist", "alpha", "k", "mean_sil", "std_sil"]]
    .sort_values("mean_sil", ascending=False)
    .reset_index(drop=True)
)

print("\n── Best (alpha, k) per distance config ──")
print(df_best.to_string(index=False))

# Gower reference
D_gower_ref = gower_lib.gower_matrix(
    df.values,
    cat_features=build_gower_cat_mask(df, categorical_cols=categorical_cols)
).astype(np.float64)

gower_sil_rows = []
for k in K_RANGE:
    sil_scores = []
    for seed in SEEDS:
        res  = kmedoids.fasterpam(D_gower_ref, k, random_state=seed)
        lbls = np.array(res.labels)
        if len(np.unique(lbls)) >= 2:
            sil_scores.append(silhouette_score(D_gower_ref, lbls, metric="precomputed"))
    if sil_scores:
        gower_sil_rows.append({
            "k": k,
            "mean_sil": np.mean(sil_scores),
            "std_sil":  np.std(sil_scores),
        })

best_gower = max(gower_sil_rows, key=lambda r: r["mean_sil"])
gower_sil  = [r["mean_sil"] for r in gower_sil_rows if r["k"] == best_gower["k"]]

logger.info("Gower best: k=%d  mean_sil=%.4f  std=%.4f",
            best_gower["k"], best_gower["mean_sil"], best_gower["std_sil"])
print(f"\nGower (best k={best_gower['k']})  "
      f"mean_sil={best_gower['mean_sil']:.4f}  std={best_gower['std_sil']:.4f}")

# Extract overall winner
winner_row   = df_best.iloc[0]
winner_num   = winner_row["num_dist"]
winner_cat   = winner_row["cat_dist"]
winner_alpha = winner_row["alpha"]
winner_k     = int(winner_row["k"])
winner_sil   = winner_row["mean_sil"]

logger.info("Winner: %s+%s  alpha=%.4f  k=%d  mean_sil=%.4f",
            winner_num, winner_cat, winner_alpha, winner_k, winner_sil)
print(f"\nWinner: {winner_num}+{winner_cat}  alpha={winner_alpha:.4f}  "
      f"k={winner_k}  mean_sil={winner_sil:.4f}")

2026-03-19 23:06:57,126 — INFO — Alpha sweep: [0.2, 0.3, 0.4, 0.5, 0.5833, 0.6, 0.8, 0.8235, 0.9]
2026-03-19 23:06:57,127 — INFO — Seeds: [42, 7, 123, 999, 2024]
2026-03-19 23:06:57,127 — INFO — Configs: 6 × alphas: 9 × k: 9 × seeds: 5 = 2430 total runs
2026-03-19 23:06:57,129 — INFO — ── Pre-flight check: numerical 'X_num[L1+Hamming]'  shape=(4950, 14) ──
2026-03-19 23:06:57,130 — INFO — [X_num[L1+Hamming]] OK   dtype: float64
2026-03-19 23:06:57,131 — INFO — [X_num[L1+Hamming]] OK   NaN: none
2026-03-19 23:06:57,132 — INFO — [X_num[L1+Hamming]] OK   Inf: none
2026-03-19 23:06:57,134 — INFO — [X_num[L1+Hamming]] OK   range [0,1]: global min=0.000000  max=1.000000
2026-03-19 23:06:57,134 — INFO — [X_num[L1+Hamming]] OK   constant: no constant columns
2026-03-19 23:06:57,136 — INFO — [X_num[L1+Hamming]] numerical pre-flight: ALL CHECKS PASSED (14 features)
2026-03-19 23:06:57,137 — INFO — ── Pre-flight check: label-encoded categorical 'X_cat[L1+Hamming]'  shape=(4950, 3) ──
2026-03-19 2

In [ ]:
# %% Cell 5 — Sweep plots
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── 1. Silhouette heatmap: alpha × k, one subplot per config ─────────────────
configs_ordered = [f"{nd}+{cd}" for nd, cd, _ in sweep_configs]

fig_heat = make_subplots(
    rows=2, cols=3,
    subplot_titles=configs_ordered,
    horizontal_spacing=0.08,
    vertical_spacing=0.14,
)

for idx, (num_dist, cat_dist, _) in enumerate(sweep_configs):
    row = idx // 3 + 1
    col = idx %  3 + 1

    subset = df_sweep[
        (df_sweep["num_dist"] == num_dist) &
        (df_sweep["cat_dist"] == cat_dist)
    ]
    pivot = subset.pivot(index="alpha", columns="k", values="mean_sil")

    fig_heat.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=[str(k) for k in pivot.columns],
            y=[str(round(a, 2)) for a in pivot.index],
            colorscale="Viridis",
            zmin=df_sweep["mean_sil"].min(),
            zmax=df_sweep["mean_sil"].max(),
            showscale=(idx == 0),
            colorbar=dict(title="Silhouette", x=1.02) if idx == 0 else None,
        ),
        row=row, col=col,
    )
    fig_heat.update_xaxes(title_text="k", row=row, col=col)
    fig_heat.update_yaxes(title_text="alpha", row=row, col=col)

fig_heat.update_layout(
    title="Mean silhouette — alpha × k heatmap per config",
    height=700,
)
fig_heat.show()

# ── 2. Best silhouette per config — bar chart ─────────────────────────────────
fig_bar = go.Figure()

fig_bar.add_trace(go.Bar(
    x=df_best["num_dist"] + "+" + df_best["cat_dist"],
    y=df_best["mean_sil"],
    error_y=dict(type="data", array=df_best["std_sil"].tolist(), visible=True),
    marker_color=px.colors.qualitative.Vivid[:len(df_best)],
    text=[f"α={row.alpha:.2f}<br>k={row.k}" for row in df_best.itertuples()],
    textposition="outside",
))

fig_bar.add_hline(
    y=float(np.mean(gower_sil)),
    line_dash="dash", line_color="gray",
    annotation_text=f"Gower ({np.mean(gower_sil):.3f})",
    annotation_position="top left",
)

fig_bar.update_layout(
    title="Best mean silhouette per config (at optimal alpha and k)",
    yaxis_title="Mean silhouette (5 seeds)",
    xaxis_title="Distance config",
    height=500,
    showlegend=False,
)
fig_bar.show()

# ── 3. Silhouette vs alpha curves ─────────────────────────────────────────────
df_alpha_curve = (
    df_sweep
    .sort_values("mean_sil", ascending=False)
    .groupby(["num_dist", "cat_dist", "alpha"])
    .first()
    .reset_index()
)

fig_curve = go.Figure()

for num_dist, cat_dist, _ in sweep_configs:
    sub = df_alpha_curve[
        (df_alpha_curve["num_dist"] == num_dist) &
        (df_alpha_curve["cat_dist"] == cat_dist)
    ].sort_values("alpha")

    fig_curve.add_trace(go.Scatter(
        x=sub["alpha"],
        y=sub["mean_sil"],
        mode="lines+markers",
        name=f"{num_dist}+{cat_dist}",
        error_y=dict(
            type="data", array=sub["std_sil"].tolist(),
            visible=True, thickness=1, width=4,
        ),
    ))

fig_curve.add_vline(
    x=alpha_neutral_hamming, line_dash="dot", line_color="gray",
    annotation_text=f"neutral Hamming ({alpha_neutral_hamming:.2f})",
    annotation_position="top right",
)
fig_curve.add_vline(
    x=alpha_neutral_tanimoto, line_dash="dot", line_color="lightgray",
    annotation_text=f"neutral Tanimoto ({alpha_neutral_tanimoto:.2f})",
    annotation_position="bottom right",
)

fig_curve.update_layout(
    title="Silhouette vs alpha — best k at each alpha",
    xaxis_title="alpha (numerical block weight)",
    yaxis_title="Mean silhouette (5 seeds)",
    height=520,
    legend=dict(x=0.01, y=0.99),
)
fig_curve.show()

In [ ]:
# %% Cell 6 — Compute all distance matrices at their optimal alpha
CACHE_DIR = Path("distance_cache")
CACHE_DIR.mkdir(exist_ok=True)
WEIGHTED_CACHE_DIR = CACHE_DIR / "weighted"
WEIGHTED_CACHE_DIR.mkdir(exist_ok=True)

def load_or_compute(path: Path, compute_fn):
    if path.exists():
        logger.info("Loading cached matrix from %s", path)
        with open(path, "rb") as f:
            return pickle.load(f)
    logger.info("Computing matrix — will save to %s", path)
    D = compute_fn()
    with open(path, "wb") as f:
        pickle.dump(D, f)
    logger.info("Saved to %s", path)
    return D

# ── All mixed configs at their sweep-optimal alpha ────────────────────────────
# df_best has one row per (num_dist, cat_dist) with the best alpha and k found
all_matrices = {}

for _, row in df_best.iterrows():
    num_dist  = row["num_dist"]
    cat_dist  = row["cat_dist"]
    opt_alpha = row["alpha"]
    X_cat     = X_cat_ohe.copy() if cat_dist == "Tanimoto" else X_cat_int.copy()
    key       = f"Mixed ({num_dist}+{cat_dist})"
    pkl_name  = f"D_mixed_{num_dist.lower()}_{cat_dist.lower()}.pkl"

    all_matrices[key] = load_or_compute(
        WEIGHTED_CACHE_DIR / pkl_name,
        lambda nd=num_dist, cd=cat_dist, xc=X_cat, a=opt_alpha: mixed_distance_matrix(
            X_num, xc,
            alpha=a,
            num_dist=nd,
            cat_dist=cd,
        )
    )
    logger.info("%-30s alpha=%.4f  shape=%s", key, opt_alpha, all_matrices[key].shape)

# ── Gower (needed for comparison notebook) ────────────────────────────────────
cat_mask = build_gower_cat_mask(df, categorical_cols=categorical_cols)
D_gower  = load_or_compute(
    WEIGHTED_CACHE_DIR / "D_gower.pkl",
    lambda: gower_lib.gower_matrix(df.values, cat_features=cat_mask)
)
all_matrices["Gower"] = D_gower

# ── Winner matrix (shortcut reference used by downstream cells) ───────────────
D_winner = all_matrices[f"Mixed ({winner_num}+{winner_cat})"]

logger.info("Gower matrix ready — shape=%s  min=%.4f  max=%.4f",
            D_gower.shape, D_gower.min(), D_gower.max())

print("\nAll matrices ready (at optimal alpha):")
for key, D in all_matrices.items():
    opt_alpha = df_best.loc[
        (df_best["num_dist"] + "+" + df_best["cat_dist"]) == key.replace("Mixed (", "").replace(")", ""),
        "alpha"
    ].values
    alpha_str = f"α={opt_alpha[0]:.4f}" if len(opt_alpha) else "—"
    print(f"  {key:35s}: {D.shape}  {alpha_str}  min={D.min():.4f}  max={D.max():.4f}")

In [ ]:
# %% Cell 7 — Sanity check on winning matrix only
logger.info("── Sanity check: %s+%s  alpha=%.4f ──",
            winner_num, winner_cat, winner_alpha)

ok = check_distance_matrix(D_winner, name=f"{winner_num}+{winner_cat}")

if not ok:
    raise RuntimeError(
        "Winning distance matrix failed sanity checks. "
        "Do not proceed to clustering."
    )

vals = D_winner[np.triu_indices(D_winner.shape[0], k=1)]
print(f"\n── {winner_num}+{winner_cat}  (alpha={winner_alpha:.4f}) ──")
print(f"  shape  : {D_winner.shape}")
print(f"  min    : {vals.min():.4f}")
print(f"  max    : {vals.max():.4f}")
print(f"  mean   : {vals.mean():.4f}")
print(f"  std    : {vals.std():.4f}")

## Alpha sweep — silhouette vs alpha curve

Every config shows a monotonically decreasing silhouette as alpha increases — clustering quality consistently improves as more weight is given to the categorical block. The optimal alpha for all six configs is at the leftmost point tested (alpha=0.2), meaning the categorical variables dominate the best-performing distance in every case.

Several observations worth noting:

**Tanimoto dominates Hamming at every alpha.** The gap between the two encoding families is consistent across the full sweep, not just at neutral alpha. This rules out the possibility that Tanimoto's advantage was an artefact of the neutral baseline — it holds regardless of weighting.

**L1+Tanimoto is the best config at every alpha.** It sits above all other curves for the entire range, confirming it as the primary candidate.

**All configs converge toward alpha=0.82–0.84 (neutral Hamming).** At the neutral Tanimoto baseline (0.58, dashed line) the spread between configs is already narrow, and by alpha=0.82 nearly all configs produce equivalent, poor silhouette scores around 0.15–0.20. This is consistent with the not-weighted results, where neutral alpha produced weak structure.

**The error bars at alpha=0.58–0.60 are visible only for L1+Tanimoto and L2+Tanimoto**, indicating those configs have slightly more variance across seeds in that region. All other configs are stable throughout.

### Conclusion

The sweep is unambiguous: categorical features drive the cluster structure in this dataset. The weighted notebook will select alpha=0.2 as optimal, giving the categorical block 80% of the distance weight. This is not a marginal improvement — silhouette at alpha=0.2 (~0.65) is more than three times higher than at the neutral Hamming baseline (~0.19).

In [ ]:
# %% Cell 8 — Cluster and visualise the winning configuration only

# Re-run kmedoids on the winning matrix at seed=42 for reproducibility
res_winner   = kmedoids.fasterpam(D_winner.astype(np.float64), winner_k, random_state=42)
winner_lbls  = np.array(res_winner.labels)

logger.info("Winner clustering: k=%d  unique clusters=%d",
            winner_k, len(np.unique(winner_lbls)))
print(f"Cluster size distribution:")
print(pd.Series(winner_lbls).value_counts().sort_index().to_string())

# UMAP 2D visualisation
reducer_2d = umap.UMAP(
    n_components=2, metric="precomputed",
    n_neighbors=min(15, D_winner.shape[0] - 1),
    min_dist=0.1, random_state=42,
)
emb2d = reducer_2d.fit_transform(D_winner.astype(np.float64))

cluster_labels = [f"Cluster {c}" for c in winner_lbls.tolist()]
df_2d = pd.DataFrame({
    "UMAP1": emb2d[:, 0], "UMAP2": emb2d[:, 1],
    "Cluster": cluster_labels,
})
fig2d = px.scatter(
    df_2d, x="UMAP1", y="UMAP2", color="Cluster",
    title=f"UMAP 2D — {winner_num}+{winner_cat}  α={winner_alpha:.2f}  k={winner_k}",
    opacity=0.6,
    color_discrete_sequence=px.colors.qualitative.Vivid,
)
fig2d.update_traces(marker=dict(size=4))
fig2d.update_layout(height=520)
fig2d.show()

# UMAP 3D visualisation
reducer_3d = umap.UMAP(
    n_components=3, metric="precomputed",
    n_neighbors=min(15, D_winner.shape[0] - 1),
    min_dist=0.1, random_state=42,
)
emb3d = reducer_3d.fit_transform(D_winner.astype(np.float64))

df_3d = pd.DataFrame({
    "UMAP1": emb3d[:, 0], "UMAP2": emb3d[:, 1], "UMAP3": emb3d[:, 2],
    "Cluster": cluster_labels,
})
fig3d = px.scatter_3d(
    df_3d, x="UMAP1", y="UMAP2", z="UMAP3", color="Cluster",
    title=f"UMAP 3D — {winner_num}+{winner_cat}  α={winner_alpha:.2f}  k={winner_k}",
    opacity=0.65,
    color_discrete_sequence=px.colors.qualitative.Vivid,
)
fig3d.update_traces(marker=dict(size=3))
fig3d.update_layout(height=620)
fig3d.show()

In [ ]:
# %% Cell — Cluster profiling (winner config only)
JOB_MAP  = {1: "Unemployed", 2: "Employee", 3: "Manager",
             4: "Entrepreneur", 5: "Retired"}
GEN_MAP  = {0: "Male", 1: "Female"}
AREA_MAP = {1: "North", 2: "Center", 3: "South"}

KEY_FEATURES     = ["Income", "Wealth", "Debt", "Saving", "Luxury", "FinEdu"]
PAL              = px.colors.qualitative.Vivid
method_name      = f"{winner_num}+{winner_cat}  α={winner_alpha:.2f}"

# Re-run kmedoids at winner config with seed=42 for reproducibility
D_f64       = D_winner.astype(np.float64)
res_winner  = kmedoids.fasterpam(D_f64, winner_k, random_state=42)
winner_lbls = np.array(res_winner.labels)

df_p = df.copy()
df_p["Cluster"]  = winner_lbls
num_profile_cols = [c for c in numerical_cols if c in df_p.columns]

# ── Cluster size distribution ─────────────────────────────────────────────────
sizes = pd.Series(winner_lbls).value_counts().sort_index()
logger.info("Cluster sizes: %s", sizes.to_dict())
print(f"\nCluster size distribution ({method_name}  k={winner_k}):")
print(sizes.to_string())
if sizes.min() < 3:
    logger.warning(
        "Cluster %d has only %d members — inspect before reporting.",
        sizes.idxmin(), sizes.min(),
    )

# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for c in range(winner_k):
    g   = df_p[df_p["Cluster"] == c]
    row = {"Cluster": c, "N": len(g), "%": f"{len(g)/len(df_p)*100:.1f}%"}
    for col in num_profile_cols:
        row[col] = round(float(g[col].mean()), 3)
    row["Job"]    = JOB_MAP.get(int(g["Job"].mode()[0]),    "?")
    row["Gender"] = GEN_MAP.get(int(g["Gender"].mode()[0]), "?")
    row["Area"]   = AREA_MAP.get(int(g["Area"].mode()[0]),  "?")
    rows.append(row)

print(f"\n{'='*60}")
print(f" {method_name} — k={winner_k} cluster summary")
print(f"{'='*60}")
display(pd.DataFrame(rows))

# ── Radar chart ───────────────────────────────────────────────────────────────
radar = go.Figure()
for c in range(winner_k):
    g    = df_p[df_p["Cluster"] == c]
    vals = [float(g[col].mean()) for col in num_profile_cols]
    radar.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=num_profile_cols + [num_profile_cols[0]],
        fill="toself", name=f"Cluster {c}",
        line_color=PAL[c % len(PAL)], opacity=0.75,
    ))
radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title=f"Radar — {method_name} (k={winner_k})",
    height=500,
)
radar.show()

# ── Boxplots for key features ─────────────────────────────────────────────────
box = make_subplots(rows=1, cols=len(KEY_FEATURES), subplot_titles=KEY_FEATURES)
for ci, col in enumerate(KEY_FEATURES):
    for c in range(winner_k):
        g = df_p[df_p["Cluster"] == c]
        box.add_trace(go.Box(
            y=g[col].tolist(), name=f"Cluster {c}",
            marker_color=PAL[c % len(PAL)], showlegend=(ci == 0),
        ), row=1, col=ci + 1)
box.update_layout(
    height=420, boxmode="group",
    title_text=f"Key features by cluster — {method_name} (k={winner_k})",
)
box.show()

In [ ]:
# %% Cell — Summary & save
summary_df = (
    df_best
    .assign(Distance=df_best["num_dist"] + "+" + df_best["cat_dist"])
    [["Distance", "alpha", "k", "mean_sil", "std_sil"]]
    .rename(columns={
        "alpha":    "Best alpha",
        "k":        "Best k",
        "mean_sil": "Mean silhouette",
        "std_sil":  "Std silhouette",
    })
    .reset_index(drop=True)
)

summary_df = pd.concat([
    summary_df,
    pd.DataFrame([{
        "Distance":        "Gower",
        "Best alpha":      None,
        "Best k":          best_gower["k"],
        "Mean silhouette": round(best_gower["mean_sil"], 4),
        "Std silhouette":  round(best_gower["std_sil"],  4),
    }])
], ignore_index=True).sort_values("Mean silhouette", ascending=False).reset_index(drop=True)

print("\nFinal comparison — best (alpha, k) per distance metric:")
display(summary_df)

print(f"\nBest performing distance : {winner_num}+{winner_cat}")
print(f"Optimal alpha            : {winner_alpha:.4f}")
print(f"Best k                   : {winner_k}")
print(f"Mean silhouette          : {winner_sil:.4f}")

# ── Save for comparison notebook ──────────────────────────────────────────────
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

pickle.dump({
    "summary_df":  summary_df,
    "winner":      f"{winner_num}+{winner_cat}",
    "best_alpha":  winner_alpha,
    "winner_k":    winner_k,
    "winner_sil":  winner_sil,
    "winner_lbls": winner_lbls,
    "df_sweep":    df_sweep,
    "df_best":     df_best,
    "best_gower":  best_gower,
}, open(RESULTS_DIR / "weighted_results.pkl", "wb"))

logger.info(
    "Saved weighted_results.pkl — winner=%s+%s  alpha=%.4f  k=%d  sil=%.4f",
    winner_num, winner_cat, winner_alpha, winner_k, winner_sil,
)
print("Saved weighted_results.pkl")